# Notebook 39b — Threat model, corrected: compression-induced blind spots and recipe-conditional lift

Notebook 39's pre-registered gate failed, and the dense-model row shows why: an exposed set defined only by
low pruned recall was dominated by the floored tier (classes no model detects), so "selection" lifted
misattribution against the dense detector as much as against the pruned one. That measures intrinsic
difficulty, not what pruning added. This notebook keeps Notebook 39 on the record and corrects the design.

**Corrected attacker knowledge.** The exposed set is the set of *compression-induced blind spots*: attack
classes that the attacker-side dense detectors catch (mean M0 recall >= 0.5) but the attacker-side
default-pruned detectors miss (mean prune80 recall < 0.2). Floored classes drop out by construction.

**Corrected claim.** The attack matters only if it is recipe-conditional: the *same* exposed traffic is
misattributed by held-out default-pruned detectors and correctly attributed by held-out detectors pruned
with a fixed recipe (first layer protected; global magnitude) or left dense. Random selection is drawn from
the same universe of baseline-detectable attack classes, so it measures the value of selection rather than
floor effects. Leave-two-seeds-out folds as before.

**Gate (stated before running).** (i) On held-out default-pruned detectors, exposed-set misattribution
exceeds 0.5 and is at least 1.5x the random-over-detectable baseline; (ii) on held-out fixed-recipe and
dense detectors, exposed-set misattribution is below 0.35 and the default-vs-fixed ratio on identical
traffic is at least 2x. Inference only; GPU runtime for speed.

In [ ]:
# --- Colab bootstrap ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys, copy, json as _json
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.utils.prune as prune
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from src.config import CFG, PATHS, set_all_seeds
from src.data import load_raw, clean, temporal_within_capture_split
from src import train as TR, models as M, explain as EXP, mitigate
from src.comnet_audit import assign_validation_tiers, calibration_summary, environment_record, write_json
from src.train import load_anchor, predict, per_class_recall_table, feature_columns

assert torch.cuda.is_available(), 'switch to a GPU runtime first'
DEVICE = TR.DEVICE
DATASET = 'ciciot2023'
SEEDS = list(CFG['seeds']); ANCHOR = int(CFG['anchor_seed'])
OUT = PATHS.tables('comnet')
PRACTICAL_LOSS = 0.10
from itertools import combinations
RECIPES = {'default_layerwise80': 'prune80_paired', 'protect_conv0': 'layerwise80_protect_conv0_paired', 'global80': 'global80_paired'}
DENSE_OK = 0.50     # attacker-side dense recall at or above this: 'baseline-detectable'
PRUNED_BAD = 0.20   # attacker-side default-pruned recall below this: 'blind spot'

ARCH = 'cnn1d'; ARCH_KW = {'channels': (64, 128)}
print('recipes:', list(RECIPES), '| blind spot: dense >=', DENSE_OK, 'and pruned <', PRUNED_BAD)

In [ ]:
df = clean(load_raw(DATASET, subsample=True, seed=ANCHOR), DATASET)
splits = temporal_within_capture_split(df, seed=ANCHOR)
feat_cols = feature_columns(df)
print(f'{len(df):,} rows | {df.label.nunique()} classes')

In [ ]:
# Per-flow test predictions for every (recipe, seed), plus the dense baselines
fam_map = pd.read_csv(OUT / 'ciciot2023_alert_family_mapping.csv').set_index('fine_label')['alert_family'].to_dict()
def load_pruned(cell, seed, le):
    m = M.build(ARCH, len(feat_cols), len(le.classes_), **ARCH_KW).to(DEVICE)
    ck = torch.load(PATHS.model(DATASET, ARCH, cell, seed), map_location=DEVICE, weights_only=False)
    m.load_state_dict(ck['state_dict'] if isinstance(ck, dict) and 'state_dict' in ck else ck); return m.eval()

preds = {}   # (recipe, seed) -> (y_true, y_pred) as class-index arrays
for seed in SEEDS:
    m0, le, scaler, _ = load_anchor(DATASET, ARCH, 'M0_paired', seed, arch_kwargs=ARCH_KW)
    yt, yp, _ = predict(m0, df, splits, le, scaler, feat_cols, which='test'); preds[('dense', seed)] = (np.asarray(yt), np.asarray(yp))
    for recipe, cell in RECIPES.items():
        mp = load_pruned(cell, seed, le)
        yt, yp, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test'); preds[(recipe, seed)] = (np.asarray(yt), np.asarray(yp))
    print(f'seed {seed}: predictions collected')
classes = list(le.classes_); C = len(classes)
benign_idx = int(np.where(np.array(classes) == 'BenignTraffic')[0][0])
family_of = np.array([fam_map[c] for c in classes])
attack_idx = [i for i in range(C) if i != benign_idx]
print(f'{C} classes, benign index {benign_idx}, {len(attack_idx)} attack classes')

In [ ]:
# Recalls, corrected exposed sets, and recipe-conditional evaluation
def per_class_recall(yt, yp):
    return np.array([(yp[yt == c] == c).mean() if (yt == c).sum() else np.nan for c in range(C)])
recall = {k: per_class_recall(*v) for k, v in preds.items()}

def attacker_sets(attacker_seeds):
    r_dense = np.nanmean([recall[('dense', s)] for s in attacker_seeds], axis=0)
    r_pruned = np.nanmean([recall[('default_layerwise80', s)] for s in attacker_seeds], axis=0)
    detectable = [c for c in attack_idx if r_dense[c] >= DENSE_OK]
    blind = [c for c in detectable if r_pruned[c] < PRUNED_BAD]
    return detectable, blind

def selection_metrics(yt, yp, chosen):
    m = np.isin(yt, chosen); t, p = yt[m], yp[m]
    if len(t) == 0: return {'n_flows': 0}
    exact = (p == t); to_benign = (p == benign_idx); cross_family = (family_of[p] != family_of[t]) & ~to_benign
    return {'n_flows': int(len(t)), 'misattribution_rate': float(1 - exact.mean()), 'attack_to_benign_rate': float(to_benign.mean()),
            'cross_family_rate': float(cross_family.mean()), 'wrong_family_or_benign_rate': float((cross_family | to_benign).mean())}

def type_uniform(yt, yp, universe):
    per = [selection_metrics(yt, yp, [c]) for c in universe]; per = [d for d in per if d['n_flows'] > 0]
    met = {k: float(np.mean([d[k] for d in per])) for k in per[0] if k != 'n_flows'}; met['n_flows'] = int(sum(d['n_flows'] for d in per)); return met

rows, set_rows = [], []
for attacker_seeds in combinations(SEEDS, 3):
    held_out = [s for s in SEEDS if s not in attacker_seeds]
    detectable, blind = attacker_sets(attacker_seeds)
    set_rows.append({'attacker_seeds': str(attacker_seeds), 'n_detectable': len(detectable), 'n_blind': len(blind), 'blind_spots': ';'.join(classes[c] for c in blind)})
    for recipe in list(RECIPES) + ['dense']:
        for seed in held_out:
            yt, yp = preds[(recipe, seed)]
            for strategy, met in (('random_over_detectable', type_uniform(yt, yp, detectable)),
                                  ('blind_spot_set', type_uniform(yt, yp, blind) if blind else {'n_flows': 0})):
                rows.append({'attacker_seeds': str(attacker_seeds), 'held_out_seed': seed, 'recipe': recipe, 'strategy': strategy,
                             'n_types': len(detectable) if strategy == 'random_over_detectable' else len(blind), **met})
fold = pd.DataFrame(rows); fold.to_csv(OUT / 'threat_model_v2_fold_results.csv', index=False)
sets = pd.DataFrame(set_rows); sets.to_csv(OUT / 'threat_model_v2_blind_spot_sets.csv', index=False)
print(sets.to_string(index=False))

In [ ]:
# Summary, recipe-conditional ratio, gate
summ = fold.groupby(['recipe', 'strategy']).agg(misattribution=('misattribution_rate', 'mean'), attack_to_benign=('attack_to_benign_rate', 'mean'),
        cross_family=('cross_family_rate', 'mean'), wrong_family_or_benign=('wrong_family_or_benign_rate', 'mean'), n_types=('n_types', 'mean')).reset_index()
def get_(recipe, strategy): return float(summ[(summ.recipe == recipe) & (summ.strategy == strategy)].misattribution.iloc[0])
sel_lift_default = get_('default_layerwise80', 'blind_spot_set') / get_('default_layerwise80', 'random_over_detectable')
ratio_vs = {r: get_('default_layerwise80', 'blind_spot_set') / get_(r, 'blind_spot_set') for r in ('protect_conv0', 'global80', 'dense')}
summ.to_csv(OUT / 'threat_model_v2_summary.csv', index=False)
print(summ.round(4).to_string(index=False))
print(f'\nselection lift on default-pruned (blind-spot set vs random over detectable): {sel_lift_default:.3f}x')
for r, v in ratio_vs.items(): print(f'recipe-conditional ratio, identical blind-spot traffic, default vs {r}: {v:.3f}x')

d_blind = get_('default_layerwise80', 'blind_spot_set')
verdict = pd.DataFrame([
 {'criterion': 'i_default_blind_spot_misattribution_gt_0.5', 'value': round(d_blind, 4), 'pass': bool(d_blind > 0.5)},
 {'criterion': 'i_selection_lift_on_default_ge_1.5x', 'value': round(sel_lift_default, 3), 'pass': bool(sel_lift_default >= 1.5)},
 {'criterion': 'ii_protect_conv0_blind_spot_misattribution_lt_0.35', 'value': round(get_('protect_conv0', 'blind_spot_set'), 4), 'pass': bool(get_('protect_conv0', 'blind_spot_set') < 0.35)},
 {'criterion': 'ii_global80_blind_spot_misattribution_lt_0.35', 'value': round(get_('global80', 'blind_spot_set'), 4), 'pass': bool(get_('global80', 'blind_spot_set') < 0.35)},
 {'criterion': 'ii_default_vs_protect_ratio_ge_2x', 'value': round(ratio_vs['protect_conv0'], 3), 'pass': bool(ratio_vs['protect_conv0'] >= 2.0)},
 {'criterion': 'ii_default_vs_global_ratio_ge_2x', 'value': round(ratio_vs['global80'], 3), 'pass': bool(ratio_vs['global80'] >= 2.0)},
 {'criterion': 'ref_default_vs_dense_ratio', 'value': round(ratio_vs['dense'], 3), 'pass': ''},
])
print(); print(verdict.to_string(index=False))
print('\nDefault recipe creates recipe-conditional blind spots that the fixed recipes close:', bool(verdict[verdict['pass'] != '']['pass'].astype(bool).all()))
verdict.to_csv(OUT / 'threat_model_v2_gate_verdict.csv', index=False)
write_json(OUT / 'threat_model_v2_environment.json', {'recipes': RECIPES, 'dense_ok': DENSE_OK, 'pruned_bad': PRUNED_BAD, 'seeds': SEEDS, 'environment': environment_record()})

In [ ]:
# --- Commit + push: main only, this notebook's own files only ---
import subprocess, shutil, glob
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert _b == 'main', f'checked-out branch is {_b!r}; run `git checkout main` first'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True)
subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred):
    shutil.copy(cred, '/root/.git-credentials'); subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
_own = 'notebooks/39b_threat_model_blind_spots_recipe_conditional.ipynb'
if os.path.exists(_own):
    d = _json.load(open(_own))
    for c in d.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/threat_model_v2_*'), check=True)
r = subprocess.run(['git', 'commit', '-m', 'notebook 39b: corrected threat model - compression-induced blind spots (dense-detectable, pruned-missed), recipe-conditional misattribution on identical traffic'], capture_output=True, text=True)
print(r.stdout or r.stderr)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed')
print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)